# RAG Temperature Experiment - Design of Experiments

**Purpose:** Test how LLM temperature affects Rule Grounding Score (RGS) in a retrieval-augmented AI agent

**Design:** Single-factor fixed-effects experiment
- **Factor:** Temperature (3 levels: 0.3, 0.5, 0.8)
- **Replicates:** 21 questions per temperature level
- **Total experimental units:** 63 (21 × 3)
- **Response variable:** RGS = C1 × (C2 + C3 + C4 + C5) / 4

**Workflow:**
1. Run experiment → CSV with responses
2. Score C1-C5 manually (or with LLM-as-judge later)
3. Calculate RGS
4. Import to JMP for ANOVA analysis

## 1. Setup and Imports

In [ ]:
# Core imports
import os
import sys
import json
import random
import time
import csv
import re
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

# API clients
import voyageai
from anthropic import Anthropic
from anthropic.types import Message

# Exception handling - with fallback for older SDK versions
try:
    from anthropic import APITimeoutError, APIConnectionError, RateLimitError
except ImportError:
    APITimeoutError = TimeoutError
    APIConnectionError = ConnectionError
    RateLimitError = Exception

# Custom retriever implementation
sys.path.append("..")
from retriever_implementation import VectorIndex, BM25Index, Retriever

print("✓ Imports successful")

In [ ]:
# Load environment and initialize clients
load_dotenv(dotenv_path='../.env')

# VoyageAI for embeddings
v_api_key = os.getenv('VOYAGE_API_KEY')
embedding_client = voyageai.Client(api_key=v_api_key)

# Anthropic for LLM
client = Anthropic()
model = os.getenv('MODEL_NAME')
max_tokens = int(os.getenv('MAX_TOKENS'))

print("✓ API clients initialized")
print(f"  Model: {model}")

In [ ]:
# Temperature levels for experiment
temp = {
    "low": 0.3,
    "medium": 0.5,
    "high": 0.8
}

print(f"✓ Temperature levels defined: {temp}")

## 2. Helper Functions

In [ ]:
# Message handling helpers
def add_user_message(messages, message):
    """Add a user message to the conversation"""
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)

def add_assistant_message(messages, message):
    """Add an assistant message to the conversation"""
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)

def chat(messages, system=None, temperature=0.5, stop_sequences=[], tools=None, timeout=60):
    """Call Claude API with timeout support"""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
        "timeout": timeout
    }
    
    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system
    
    return client.messages.create(**params)

def text_from_message(message):
    """Extract text content from Claude response"""
    return "\n".join([block.text for block in message.content if block.type == "text"])

print("✓ Helper functions defined")

## 3. RAG Setup

In [ ]:
# Document chunking
def chunk_by_section(document_text):
    """Split document by ## headers"""
    pattern = r"\n## "
    return re.split(pattern, document_text)

print("✓ Chunking function defined")

In [ ]:
# Embedding generation
def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    """Generate embeddings using VoyageAI"""
    is_list = isinstance(chunks, list)
    input_data = chunks if is_list else [chunks]
    result = embedding_client.embed(input_data, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

print("✓ Embedding function defined")

In [ ]:
# Reranker
def reranker_fn(docs, query_text, k):
    """Use Claude to rerank retrieved documents"""
    joined_docs = "\n".join([
        f"""
        <document>
        <document_id>{doc["id"]}</document_id>
        <document_content>{doc["content"]}</document_content>
        </document>
        """
        for doc in docs
    ])
    
    prompt = f"""
    You are about to be given a set of documents, along with an id of each.
    Your task is to select the {k} most relevant documents to answer the user's question.

    Here is the user's question:
    <question>
    {query_text}
    </question>
    
    Here are the documents to select from:
    <documents>
    {joined_docs}
    </documents>

    Respond in the following format:
    ```json
    {{
        "document_ids": str[]
    }}
    ```
    """
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    
    result = chat(messages, stop_sequences=["```"])
    return json.loads(text_from_message(result))["document_ids"]

print("✓ Reranker function defined")

In [ ]:
# Contextual chunk enhancement
def add_context(text_chunk, source_text):
    """Add context to a chunk for better retrieval"""
    prompt = f"""
    Write a short and succinct snippet of text to situate this chunk within the 
    overall source document for the purposes of improving search retrieval of the chunk. 

    Here is the original source document:
    <document> 
    {source_text}
    </document> 

    Here is the chunk we want to situate within the whole document:
    <chunk> 
    {text_chunk}
    </chunk>
    
    Answer only with the succinct context and nothing else. 
    """
    
    messages = []
    add_user_message(messages, prompt)
    result = chat(messages)
    
    return text_from_message(result) + "\n" + text_chunk

print("✓ Context enhancement function defined")

In [ ]:
# Load and process the Clue rules document
with open("../References/ClueRules.md", "r") as f:
    clue_instructions = f.read()

chunks = chunk_by_section(clue_instructions)
print(f"✓ Loaded Clue rules and created {len(chunks)} chunks")

In [ ]:
# Create retriever
vector_index = VectorIndex(embedding_fn=generate_embedding)
bm25_index = BM25Index()
retriever = Retriever(bm25_index, vector_index, reranker_fn=reranker_fn)

print("✓ Retriever created")

In [ ]:
# Add contextualized chunks to retriever
num_start_chunks = 2
num_prev_chunks = 2
contextualized_chunks = []

print("Adding contextualized chunks to retriever...")
for i, chunk in enumerate(chunks):
    context_parts = []
    context_parts.extend(chunks[: min(num_start_chunks, len(chunks))])
    start_idx = max(0, i - num_prev_chunks)
    context_parts.extend(chunks[start_idx:i])
    context = "\n".join(context_parts)
    
    contextualized_chunks.append(add_context(chunk, context))
    print(f"  Processed chunk {i+1}/{len(chunks)}")

retriever.add_documents([{"content": chunk} for chunk in contextualized_chunks])
print(f"✓ Added {len(contextualized_chunks)} contextualized chunks to retriever")

## 4. RAG Tool Definition

In [ ]:
# Search function with citation metadata
def search_clue_instructions(query, k=3):
    """
    Search the Clue game instructions with citation metadata.
    
    Args:
        query (str): Search query
        k (int): Number of chunks to return
    
    Returns:
        str: JSON string with search results
    """
    results = retriever.search(query, k=k)
    
    formatted_results = []
    for i, (doc, score) in enumerate(results):
        chunk_content = doc["content"]
        preview = chunk_content[:200].strip()
        if len(chunk_content) > 200:
            preview += "..."
        
        formatted_results.append({
            "chunk_id": f"CHUNK_{i+1}",
            "content": chunk_content,
            "preview": preview,
            "relevance_score": float(score)
        })
    
    return json.dumps(formatted_results, indent=2)

print("✓ Search function defined")

In [ ]:
# Tool schema for Claude
tools = [
    {
        "name": "search_clue_instructions",
        "description": "Search the official Clue game instructions to find information about rules, gameplay, setup, winning conditions, and player counts. Use this tool whenever you need specific information from the Clue game manual.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query - what information to look for in the Clue instructions (e.g., 'player count', 'how to win', 'setup rules')"
                }
            },
            "required": ["query"]
        }
    }
]

print("✓ Tool schema defined")

## 5. Agent with Retry Logic

In [ ]:
def answer_clue_question(user_question, temperature_value=0.5, max_iterations=5, max_retries=3, retry_delay=5):
    """
    Answer a question about Clue using RAG via tool use.
    Includes timeout and retry logic.
    
    Args:
        user_question (str): The user's question
        temperature_value (float): Temperature for this run
        max_iterations (int): Max tool-use cycles
        max_retries (int): Max retry attempts on errors
        retry_delay (int): Seconds between retries
    
    Returns:
        dict: {"answer": str, "iterations": int, "retries": int, "error": str or None, "tool_calls": list}
    """
    
    system_prompt = """
<role>
You are a fun and enthusiastic game instructor who teaches people how to play Clue.
</role>

<capabilities>
<can_answer>
- Rules and game mechanics
- Setup instructions
- Turn structure
- Win conditions
- Card and token mechanics
- Suggestions and accusations
</can_answer>

<cannot_answer>
- Strategy advice
- Probability calculations
- Tactical recommendations
- Player psychology
</cannot_answer>
</capabilities>

<tools>
<tool_name>search_clue_instructions</tool_name>
<when_to_use>Whenever you need specific information from the official Clue game rules</when_to_use>
</tools>

<response_format>
<for_rules_questions>
1. Use search_clue_instructions tool
2. Provide clear, friendly explanation
3. Cite sources: "According to CHUNK_X..."
4. Include brief direct quote
</for_rules_questions>

<for_strategy_questions>
Politely decline and redirect
</for_strategy_questions>

<for_off_topic_questions>
"I only answer questions about the game Clue."
</for_off_topic_questions>
</response_format>
"""
    
    retry_count = 0
    
    while retry_count <= max_retries:
        try:
            messages = []
            add_user_message(messages, user_question)
            iteration_count = 0
            tool_calls = []
            
            for iteration in range(max_iterations):
                iteration_count = iteration + 1
                print(f"--- Iteration {iteration + 1} ---")
                
                response = chat(
                    messages=messages,
                    system=system_prompt,
                    temperature=temperature_value,
                    tools=tools,
                    timeout=60
                )
                
                print(f"Stop reason: {response.stop_reason}")
                
                if response.stop_reason == "end_turn":
                    return {
                        "answer": text_from_message(response),
                        "iterations": iteration_count,
                        "retries": retry_count,
                        "error": None,
                        "tool_calls": tool_calls
                    }
                
                if response.stop_reason == "tool_use":
                    add_assistant_message(messages, response)
                    
                    for content_block in response.content:
                        if content_block.type == "tool_use":
                            tool_name = content_block.name
                            tool_input = content_block.input
                            tool_use_id = content_block.id
                            
                            tool_calls.append({
                                "tool": tool_name,
                                "query": tool_input.get("query", "")
                            })
                            
                            print(f"Claude wants to use: {tool_name}")
                            print(f"With query: {tool_input['query']}")
                            
                            try:
                                if tool_name == "search_clue_instructions":
                                    tool_result = search_clue_instructions(tool_input["query"])
                                else:
                                    tool_result = json.dumps({"error": f"Unknown tool: {tool_name}"})
                                
                                print(f"Tool result preview: {tool_result[:150]}...")
                            
                            except Exception as tool_error:
                                print(f"⚠️ Tool execution error: {str(tool_error)[:100]}")
                                tool_result = json.dumps({
                                    "error": f"Tool execution failed: {str(tool_error)}",
                                    "tool": tool_name
                                })
                            
                            messages.append({
                                "role": "user",
                                "content": [{
                                    "type": "tool_result",
                                    "tool_use_id": tool_use_id,
                                    "content": tool_result
                                }]
                            })
                    
                    continue
                
                return {
                    "answer": text_from_message(response),
                    "iterations": iteration_count,
                    "retries": retry_count,
                    "error": None,
                    "tool_calls": tool_calls
                }
            
            return {
                "answer": "I apologize, but I'm having trouble processing this question.",
                "iterations": iteration_count,
                "retries": retry_count,
                "error": "max_iterations_reached",
                "tool_calls": tool_calls
            }
        
        except (APITimeoutError, APIConnectionError, TimeoutError, ConnectionError) as e:
            error_name = type(e).__name__.lower()
            error_type = "timeout" if "timeout" in error_name else "connection"
            
            if iteration_count > 0:
                print(f"❌ {error_type.capitalize()} error mid-conversation - cannot retry")
                return {
                    "answer": "",
                    "iterations": iteration_count,
                    "retries": retry_count,
                    "error": f"{error_type}_mid_conversation: {str(e)}",
                    "tool_calls": tool_calls if 'tool_calls' in locals() else []
                }
            
            retry_count += 1
            
            if retry_count <= max_retries:
                print(f"⚠️ {error_type.capitalize()} error (attempt {retry_count}/{max_retries+1})")
                print(f"   Waiting {retry_delay}s...")
                time.sleep(retry_delay)
            else:
                print(f"❌ Max retries exceeded")
                return {
                    "answer": "",
                    "iterations": 0,
                    "retries": retry_count - 1,
                    "error": f"{error_type}_error: {str(e)}",
                    "tool_calls": []
                }
        
        except RateLimitError as e:
            print(f"❌ Rate limit error: {str(e)}")
            return {
                "answer": "",
                "iterations": 0,
                "retries": retry_count,
                "error": f"rate_limit_error: {str(e)}",
                "tool_calls": []
            }
        
        except Exception as e:
            error_name = type(e).__name__.lower()
            if "ratelimit" in error_name or "rate_limit" in error_name:
                print(f"❌ Rate limit error: {str(e)}")
                return {
                    "answer": "",
                    "iterations": 0,
                    "retries": retry_count,
                    "error": f"rate_limit_error: {str(e)}",
                    "tool_calls": []
                }
            
            print(f"❌ Unexpected error: {str(e)}")
            return {
                "answer": "",
                "iterations": 0,
                "retries": retry_count,
                "error": f"unexpected_error: {str(e)}",
                "tool_calls": []
            }

print("✓ Agent function defined with timeout/retry logic")

## 6. Load Experimental Questions

In [ ]:
def load_questions_from_csv(csv_path):
    """
    Load questions with their types from CSV.
    
    Returns:
        list: [{"number": 1, "question": "...", "type": "S"}, ...]
    """
    df = pd.read_csv(csv_path)
    
    questions = []
    for _, row in df.iterrows():
        questions.append({
            "number": int(row['Number']),
            "question": row['Question'],
            "type": row['Type']
        })
    
    print(f"✅ Loaded {len(questions)} questions from {csv_path}")
    
    type_counts = df['Type'].value_counts().to_dict()
    print(f"   Question types: S={type_counts.get('S', 0)}, "
          f"M={type_counts.get('M', 0)}, F={type_counts.get('F', 0)}")
    
    return questions

# Load questions
questions = load_questions_from_csv("../References/ClueQuestions.csv")

## 7. Temperature Experiment Function

In [ ]:
def run_temperature_experiment(
    questions_data,
    temperature_dict,
    csv_filename="clue_temperature_experiment.csv",
    delay_seconds=3,
    randomize_order=True
):
    """
    Run temperature DOE experiment.
    
    Args:
        questions_data (list): Questions from load_questions_from_csv()
        temperature_dict (dict): Temperature levels to test
        csv_filename (str): Output CSV file
        delay_seconds (int): Delay between API calls
        randomize_order (bool): Randomize question order within each temperature
    
    Returns:
        dict: Experiment statistics
    """
    
    stats = {
        "start_time": datetime.now(),
        "total_runs": 0,
        "successful_runs": 0,
        "failed_runs": 0,
        "by_temperature": {}
    }
    
    with open(csv_filename, 'a', newline='', encoding='utf-8') as csvfile:
        fieldnames = [
            'run_id', 'timestamp',
            'temperature_level', 'temperature_value',
            'question_number', 'question_type', 'question_text',
            'answer',
            'status', 'error_message', 'iteration_count', 'retry_count',
            'tool_calls_count', 'search_queries',
            'C1_correctness', 'C2_coverage', 'C3_citation_presence',
            'C4_citation_valid', 'C5_clarity', 'RGS',
            'scored_by', 'scoring_notes'
        ]
        
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        if csvfile.tell() == 0:
            writer.writeheader()
            print(f"📊 New experiment: {csv_filename}")
        else:
            print(f"📊 Appending to: {csv_filename}")
        
        print("\n" + "="*70)
        print("TEMPERATURE EXPERIMENT - DOE")
        print("="*70)
        print(f"Factor: Temperature ({len(temperature_dict)} levels)")
        print(f"Replicates: {len(questions_data)} questions per level")
        print(f"Total runs: {len(temperature_dict) * len(questions_data)}")
        print(f"RGS Formula: C1 × (C2+C3+C4+C5) / 4")
        print("="*70 + "\n")
        
        run_id = 0
        
        for temp_label, temp_value in temperature_dict.items():
            stats["by_temperature"][temp_label] = {"successful": 0, "failed": 0}
            
            print(f"\n{'='*70}")
            print(f"🌡️  TEMPERATURE: {temp_label.upper()} (T = {temp_value})")
            print(f"{'='*70}\n")
            
            questions_for_temp = questions_data.copy()
            if randomize_order:
                random.shuffle(questions_for_temp)
                print(f"🔀 Questions randomized\n")
            
            for q_idx, q_data in enumerate(questions_for_temp, 1):
                run_id += 1
                stats["total_runs"] += 1
                
                question_text = q_data["question"]
                question_type = q_data["type"]
                question_number = q_data["number"]
                
                print(f"Run {run_id:2d} | T={temp_value} | Q{question_number:2d} | Type={question_type}")
                print(f"Q: {question_text[:55]}...")
                
                timestamp = datetime.now().isoformat()
                
                result = answer_clue_question(
                    question_text,
                    temperature_value=temp_value,
                    max_iterations=5,
                    max_retries=3,
                    retry_delay=5
                )
                
                if result["error"] is None:
                    status = "SUCCESS"
                    stats["successful_runs"] += 1
                    stats["by_temperature"][temp_label]["successful"] += 1
                    print(f"✅ Success")
                    print(f"A: {result['answer'][:60]}...")
                else:
                    status = "ERROR"
                    stats["failed_runs"] += 1
                    stats["by_temperature"][temp_label]["failed"] += 1
                    print(f"❌ Error: {result['error'][:60]}...")
                
                writer.writerow({
                    'run_id': run_id,
                    'timestamp': timestamp,
                    'temperature_level': temp_label,
                    'temperature_value': temp_value,
                    'question_number': question_number,
                    'question_type': question_type,
                    'question_text': question_text,
                    'answer': result["answer"],
                    'status': status,
                    'error_message': result["error"] or '',
                    'iteration_count': result["iterations"],
                    'retry_count': result["retries"],
                    'tool_calls_count': len(result.get("tool_calls", [])),
                    'search_queries': "; ".join([tc["query"] for tc in result.get("tool_calls", [])]),
                    'C1_correctness': '',
                    'C2_coverage': '',
                    'C3_citation_presence': '',
                    'C4_citation_valid': '',
                    'C5_clarity': '',
                    'RGS': '',
                    'scored_by': '',
                    'scoring_notes': ''
                })
                
                if not (temp_label == list(temperature_dict.keys())[-1] and 
                        q_idx == len(questions_for_temp)):
                    print(f"⏳ {delay_seconds}s...\n")
                    time.sleep(delay_seconds)
                else:
                    print()
    
    stats["end_time"] = datetime.now()
    stats["duration_minutes"] = (stats["end_time"] - stats["start_time"]).total_seconds() / 60
    
    print("\n" + "="*70)
    print("EXPERIMENT COMPLETE")
    print("="*70)
    print(f"Total: {stats['total_runs']}")
    print(f"Success: {stats['successful_runs']}")
    print(f"Failed: {stats['failed_runs']}")
    print(f"Duration: {stats['duration_minutes']:.1f} min")
    print(f"\nBy temperature:")
    for temp_label in temperature_dict.keys():
        temp_stats = stats["by_temperature"][temp_label]
        print(f"  {temp_label:8s}: {temp_stats['successful']:2d} ✓, {temp_stats['failed']:2d} ✗")
    print("="*70)
    print(f"\n📁 {csv_filename}")
    print(f"📊 Next: Score C1-C5, then analyze in JMP\n")
    
    return stats

print("✓ Experiment function defined")

## 8. Post-Experiment Helper Functions

In [ ]:
def create_scoring_template(input_csv, output_csv="scoring_template.csv"):
    """
    Create simplified CSV for manual scoring.
    """
    df = pd.read_csv(input_csv)
    df_success = df[df['status'] == 'SUCCESS'].copy()
    
    scoring_df = df_success[[
        'run_id', 'question_number', 'question_type',
        'question_text', 'temperature_value', 'answer'
    ]].copy()
    
    scoring_df['C1_correctness'] = ''
    scoring_df['C2_coverage'] = ''
    scoring_df['C3_citation_presence'] = ''
    scoring_df['C4_citation_valid'] = ''
    scoring_df['C5_clarity'] = ''
    scoring_df['scoring_notes'] = ''
    
    scoring_df.to_csv(output_csv, index=False)
    print(f"✅ Scoring template: {output_csv}")
    print(f"   {len(scoring_df)} responses to score")

def merge_scores(experiment_csv, scored_csv, output_csv="final_with_rgs.csv"):
    """
    Merge scored C1-C5 back and calculate RGS.
    RGS = C1 × (C2 + C3 + C4 + C5) / 4
    """
    df_exp = pd.read_csv(experiment_csv)
    df_scored = pd.read_csv(scored_csv)
    
    df_scored['RGS'] = (
        df_scored['C1_correctness'] * 
        (df_scored['C2_coverage'] + df_scored['C3_citation_presence'] + 
         df_scored['C4_citation_valid'] + df_scored['C5_clarity']) / 4
    )
    
    df_merged = df_exp.merge(
        df_scored[['run_id', 'C1_correctness', 'C2_coverage',
                   'C3_citation_presence', 'C4_citation_valid',
                   'C5_clarity', 'RGS', 'scoring_notes']],
        on='run_id', how='left', suffixes=('', '_scored')
    )
    
    for col in ['C1_correctness', 'C2_coverage', 'C3_citation_presence',
                'C4_citation_valid', 'C5_clarity', 'RGS', 'scoring_notes']:
        if f'{col}_scored' in df_merged.columns:
            df_merged[col] = df_merged[f'{col}_scored'].combine_first(df_merged[col])
            df_merged.drop(f'{col}_scored', axis=1, inplace=True)
    
    df_merged.to_csv(output_csv, index=False)
    print(f"✅ Final CSV: {output_csv}")
    print(f"   RGS stats:\n{df_merged[df_merged['status']=='SUCCESS']['RGS'].describe()}")

print("✓ Post-experiment helpers defined")

## 9. RUN EXPERIMENT

**Before running:**
- Ensure all previous cells have run successfully
- Check that `questions` variable is defined (21 questions loaded)
- Verify API keys are working

**Expected runtime:** ~5-7 minutes (63 runs × 3s delay + processing time)

In [ ]:
# Run the experiment
stats = run_temperature_experiment(
    questions_data=questions,
    temperature_dict=temp,
    csv_filename="clue_temperature_experiment.csv",
    delay_seconds=3,
    randomize_order=True
)

## 10. Post-Experiment Workflow

After experiment completes:

1. **Create scoring template**
2. **Score responses** (manually or with LLM-as-judge)
3. **Merge scores** and calculate RGS
4. **Import to JMP** for ANOVA

In [ ]:
# Step 1: Create scoring template
create_scoring_template(
    input_csv="clue_temperature_experiment.csv",
    output_csv="responses_to_score.csv"
)

In [ ]:
# Step 2: After scoring in Excel, merge back
merge_scores(
    experiment_csv="clue_temperature_experiment.csv",
    scored_csv="responses_to_score.csv",
    output_csv="final_experiment_with_rgs.csv"
)

# Import final_experiment_with_rgs.csv into JMP for analysis!